# SLURM Resource Efficiency Analysis

Aggregates per-job SLURM efficiency reports from MosaiCatcher pipeline runs to identify
resource optimization opportunities. Analyzes CPU efficiency, memory usage, and runtime
across all rules to suggest better resource allocations.

In [ ]:
import glob
import re
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Truncate long rule names for axis labels, full name shown on hover
def truncate(name, maxlen=35):
    return name if len(name) <= maxlen else name[:maxlen-1] + "\u2026"

# Load all CSV files
csv_dir = "/scratch/tweber/mosaicatcher_logs/slurm_efficiency_reports/"
csv_files = sorted(glob.glob(f"{csv_dir}/*.csv"))
print(f"Found {len(csv_files)} report files")

dfs = []
for f in csv_files:
    tmp = pd.read_csv(f)
    tmp["RunID"] = re.search(r"efficiency_report_(.+)\.csv", f).group(1)
    dfs.append(tmp)

df = pd.concat(dfs, ignore_index=True)

# Parse base rule name: rule_<name>_wildcards_... -> <name>
def parse_rule_name(s):
    if pd.isna(s):
        return "unknown"
    m = re.match(r"rule_(.+?)_wildcards_", str(s))
    return m.group(1) if m else str(s)

df["Rule"] = df["RuleName"].apply(parse_rule_name)

# Clean up numeric columns
for col in ["CPU Efficiency (%)", "Memory Usage (%)", "MaxRSS_MB", "RequestedMem_MB", "Elapsed_sec", "TotalCPU_sec"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"Total jobs: {len(df):,}")
print(f"Unique runs: {df['RunID'].nunique()}")
print(f"Unique rules: {df['Rule'].nunique()}")

## Overview: Efficiency Distributions

In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=(
    "CPU Efficiency Distribution", "Memory Usage Distribution", "Elapsed Time Distribution"
))

cpu_med = df["CPU Efficiency (%)"].median()
mem_med = df["Memory Usage (%)"].median()

fig.add_trace(go.Histogram(x=df["CPU Efficiency (%)"].dropna(), nbinsx=50,
    marker_color="steelblue", name="CPU Eff"), row=1, col=1)
fig.add_vline(x=cpu_med, line_dash="dash", line_color="red",
    annotation_text=f"Median: {cpu_med:.1f}%", row=1, col=1)

fig.add_trace(go.Histogram(x=df["Memory Usage (%)"].dropna(), nbinsx=50,
    marker_color="darkorange", name="Mem Use"), row=1, col=2)
fig.add_vline(x=mem_med, line_dash="dash", line_color="red",
    annotation_text=f"Median: {mem_med:.1f}%", row=1, col=2)

elapsed = df["Elapsed_sec"].dropna()
fig.add_trace(go.Histogram(x=elapsed[elapsed > 0], nbinsx=50,
    marker_color="seagreen", name="Elapsed"), row=1, col=3)
fig.update_yaxes(type="log", row=1, col=3)

fig.update_layout(height=400, showlegend=False, template="plotly_white")
fig.update_xaxes(title_text="CPU Efficiency (%)", row=1, col=1)
fig.update_xaxes(title_text="Memory Usage (%)", row=1, col=2)
fig.update_xaxes(title_text="Elapsed (seconds)", row=1, col=3)
fig.show()

print(f"Median CPU Efficiency: {cpu_med:.1f}%")
print(f"Median Memory Usage:   {mem_med:.1f}%")
print(f"Jobs with <10% CPU eff: {(df['CPU Efficiency (%)'] < 10).sum()} ({(df['CPU Efficiency (%)'] < 10).mean()*100:.1f}%)")
print(f"Jobs with <10% mem use: {(df['Memory Usage (%)'] < 10).sum()} ({(df['Memory Usage (%)'] < 10).mean()*100:.1f}%)")

## Per-Rule Aggregation

In [ ]:
rule_stats = df.groupby("Rule").agg(
    count=("Rule", "size"),
    cpu_eff_mean=("CPU Efficiency (%)", "mean"),
    cpu_eff_median=("CPU Efficiency (%)", "median"),
    mem_use_mean=("Memory Usage (%)", "mean"),
    mem_use_median=("Memory Usage (%)", "median"),
    mem_actual_mean=("MaxRSS_MB", "mean"),
    mem_actual_p95=("MaxRSS_MB", lambda x: x.quantile(0.95)),
    mem_requested_mean=("RequestedMem_MB", "mean"),
    elapsed_mean=("Elapsed_sec", "mean"),
    elapsed_median=("Elapsed_sec", "median"),
    elapsed_p95=("Elapsed_sec", lambda x: x.quantile(0.95)),
).reset_index()

rule_stats["wasted_mem_total_MB"] = (
    (rule_stats["mem_requested_mean"] - rule_stats["mem_actual_mean"]) * rule_stats["count"]
)
rule_stats = rule_stats.sort_values("wasted_mem_total_MB", ascending=False)

# Interactive plotly table for top 25 rules
top25 = rule_stats.head(25)

wasted_max = top25["wasted_mem_total_MB"].max() if len(top25) > 0 else 1
wasted_colors = []
for v in top25["wasted_mem_total_MB"]:
    shade = max(0, 255 - int(v / wasted_max * 200))
    wasted_colors.append(f"rgba(255,{shade},{shade},0.7)")

fig = go.Figure(data=[go.Table(
    columnwidth=[250, 50, 70, 70, 80, 80, 80, 70, 100],
    header=dict(
        values=["Rule", "Count", "CPU Eff<br>Median", "Mem Use<br>Median",
                "Actual Mem<br>Mean (MB)", "Actual Mem<br>p95 (MB)", "Requested<br>Mem (MB)",
                "Elapsed<br>Median (s)", "Wasted Mem<br>Total (MB)"],
        fill_color="#2c3e50", font=dict(color="white", size=11),
        align="left"
    ),
    cells=dict(
        values=[
            top25["Rule"],
            top25["count"],
            top25["cpu_eff_median"].round(1).astype(str) + "%",
            top25["mem_use_median"].round(1).astype(str) + "%",
            top25["mem_actual_mean"].round(0).astype(int),
            top25["mem_actual_p95"].round(0).astype(int),
            top25["mem_requested_mean"].round(0).astype(int),
            top25["elapsed_median"].round(0).astype(int),
            top25["wasted_mem_total_MB"].round(0).astype(int),
        ],
        fill_color=[
            "white", "white", "white", "white", "white", "white", "white", "white",
            wasted_colors
        ],
        align="left", font=dict(size=11)
    )
)])
fig.update_layout(title="Top 25 Rules by Total Wasted Memory", height=700, template="plotly_white")
fig.show()

## Top Offenders: Memory Over-Provisioning & CPU Waste

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "Top 15 Rules: Memory Over-Provisioning",
    "Top 15 Rules: Worst CPU Efficiency (min 5 jobs)"
), horizontal_spacing=0.3)

# Memory over-provisioning
top_mem = rule_stats.nlargest(15, "wasted_mem_total_MB").iloc[::-1]
fig.add_trace(go.Bar(
    y=[truncate(r) for r in top_mem["Rule"]],
    x=top_mem["wasted_mem_total_MB"] / 1000,
    orientation="h", marker_color="darkorange", name="Wasted Memory",
    customdata=top_mem["Rule"],
    hovertemplate="<b>%{customdata}</b><br>Wasted: %{x:.1f} GB<extra></extra>"
), row=1, col=1)
fig.update_xaxes(title_text="Total Wasted Memory (GB)", row=1, col=1)

# Worst CPU efficiency
freq_rules = rule_stats[rule_stats["count"] >= 5]
top_cpu = freq_rules.nsmallest(15, "cpu_eff_median").iloc[::-1]
colors = ["firebrick" if v < 10 else "darkorange" if v < 25 else "gold" for v in top_cpu["cpu_eff_median"]]
fig.add_trace(go.Bar(
    y=[truncate(r) for r in top_cpu["Rule"]],
    x=top_cpu["cpu_eff_median"],
    orientation="h", marker_color=colors, name="CPU Eff",
    customdata=top_cpu["Rule"],
    hovertemplate="<b>%{customdata}</b><br>CPU Eff: %{x:.1f}%<extra></extra>"
), row=1, col=2)
fig.add_vline(x=50, line_dash="dash", line_color="gray", opacity=0.5, row=1, col=2)
fig.update_xaxes(title_text="Median CPU Efficiency (%)", row=1, col=2)

fig.update_layout(height=600, showlegend=False, template="plotly_white")
fig.show()

## Memory Optimization: Unused Memory Distribution per Rule

In [ ]:
SAFETY_MARGIN = 1.3  # 30% headroom above p95
MIN_MEMORY_MB = 500  # Don't go below profile default

# Compute per-job unused memory
df["unused_mem_MB"] = df["RequestedMem_MB"] - df["MaxRSS_MB"]
df["unused_mem_pct"] = 100 - df["Memory Usage (%)"]

# Pre-compute mem_opt for the recommendations cell
mem_opt = rule_stats[["Rule", "count", "mem_actual_mean", "mem_actual_p95", "mem_requested_mean"]].copy()
mem_opt["suggested_MB"] = np.maximum(mem_opt["mem_actual_p95"] * SAFETY_MARGIN, MIN_MEMORY_MB).round(-1)
mem_opt["savings_per_job_MB"] = mem_opt["mem_requested_mean"] - mem_opt["suggested_MB"]
mem_opt["total_savings_MB"] = mem_opt["savings_per_job_MB"] * mem_opt["count"]
mem_opt = mem_opt.sort_values("total_savings_MB", ascending=False)

# Order rules by median unused memory (descending)
rule_order = (
    df.groupby("Rule")["unused_mem_MB"]
    .median()
    .sort_values(ascending=True)
    .index.tolist()
)
# Filter to rules with >= 5 jobs for readability
freq_rules_list = df.groupby("Rule").filter(lambda g: len(g) >= 5)["Rule"].unique()
plot_df = df[df["Rule"].isin(freq_rules_list)].copy()

fig = go.Figure()
for rule in [r for r in rule_order if r in freq_rules_list]:
    subset = plot_df[plot_df["Rule"] == rule]["unused_mem_MB"].dropna()
    fig.add_trace(go.Box(
        x=subset,
        y=[truncate(rule)] * len(subset),
        name=truncate(rule),
        orientation="h",
        marker_color="darkorange",
        line_color="firebrick",
        customdata=[rule] * len(subset),
        hovertemplate="<b>%{customdata}</b><br>Unused: %{x:.0f} MB<extra></extra>",
        showlegend=False,
    ))

fig.update_layout(
    height=max(500, len(freq_rules_list) * 25 + 100),
    template="plotly_white",
    title="Distribution of Unused Memory (Requested - Actual) per Rule (min 5 jobs)",
    xaxis_title="Unused Memory (MB)",
    yaxis=dict(dtick=1),
)
fig.show()

# Summary stats
med_unused = df["unused_mem_MB"].median()
mean_unused = df["unused_mem_MB"].mean()
total_unused_gb = df["unused_mem_MB"].sum() / 1000
print(f"\nPer-job unused memory: median={med_unused:.0f} MB, mean={mean_unused:.0f} MB")
print(f"Total unused memory across all {len(df):,} jobs: {total_unused_gb:,.1f} GB")

## CPU Efficiency Deep Dive

Analyzes per-rule CPU utilization: `CPU Efficiency = TotalCPU_sec / (Elapsed_sec × NCPUS)`.
Rules requesting multiple CPUs but achieving low efficiency are candidates for reducing `threads`.

In [ ]:
# Ensure NCPUS is numeric
df["NCPUS"] = pd.to_numeric(df["NCPUS"], errors="coerce").fillna(1).astype(int)

# Per-rule CPU stats
cpu_stats = df.groupby("Rule").agg(
    count=("Rule", "size"),
    ncpus_mode=("NCPUS", lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 1),
    cpu_eff_mean=("CPU Efficiency (%)", "mean"),
    cpu_eff_median=("CPU Efficiency (%)", "median"),
    cpu_eff_p5=("CPU Efficiency (%)", lambda x: x.quantile(0.05)),
    total_cpu_sec=("TotalCPU_sec", "sum"),
    total_elapsed_sec=("Elapsed_sec", "sum"),
).reset_index()

# Wasted CPU-seconds: (Elapsed × NCPUS) - TotalCPU for each job
df["allocated_cpu_sec"] = df["Elapsed_sec"] * df["NCPUS"]
df["wasted_cpu_sec"] = df["allocated_cpu_sec"] - df["TotalCPU_sec"]
cpu_waste_per_rule = df.groupby("Rule")["wasted_cpu_sec"].sum().reset_index()
cpu_stats = cpu_stats.merge(cpu_waste_per_rule, on="Rule")
cpu_stats["wasted_cpu_hours"] = cpu_stats["wasted_cpu_sec"] / 3600
cpu_stats = cpu_stats.sort_values("wasted_cpu_hours", ascending=False)

# --- Plot 1: CPU efficiency distribution per rule (box plot) ---
freq_cpu_rules = df.groupby("Rule").filter(lambda g: len(g) >= 5)["Rule"].unique()
cpu_rule_order = (
    df[df["Rule"].isin(freq_cpu_rules)]
    .groupby("Rule")["CPU Efficiency (%)"]
    .median()
    .sort_values(ascending=True)
    .index.tolist()
)

fig = go.Figure()
for rule in cpu_rule_order:
    subset = df[(df["Rule"] == rule) & df["CPU Efficiency (%)"].notna()]["CPU Efficiency (%)"]
    fig.add_trace(go.Box(
        x=subset,
        y=[truncate(rule)] * len(subset),
        name=truncate(rule),
        orientation="h",
        marker_color="steelblue",
        line_color="midnightblue",
        customdata=[rule] * len(subset),
        hovertemplate="<b>%{customdata}</b><br>CPU Eff: %{x:.1f}%<extra></extra>",
        showlegend=False,
    ))

fig.add_vline(x=50, line_dash="dash", line_color="red", opacity=0.5,
    annotation_text="50% threshold")
fig.update_layout(
    height=max(500, len(freq_cpu_rules) * 25 + 100),
    template="plotly_white",
    title="CPU Efficiency Distribution per Rule (min 5 jobs)",
    xaxis_title="CPU Efficiency (%)",
    yaxis=dict(dtick=1),
)
fig.show()

# --- Plot 2: Multi-CPU rules — requested vs utilized CPUs ---
multi_cpu = df[df["NCPUS"] > 1].copy()
if len(multi_cpu) > 0:
    multi_cpu["effective_cpus"] = multi_cpu["TotalCPU_sec"] / multi_cpu["Elapsed_sec"].replace(0, np.nan)
    multi_stats = multi_cpu.groupby("Rule").agg(
        count=("Rule", "size"),
        ncpus=("NCPUS", "first"),
        effective_cpus_mean=("effective_cpus", "mean"),
        effective_cpus_median=("effective_cpus", "median"),
        cpu_eff_median=("CPU Efficiency (%)", "median"),
    ).reset_index().sort_values("cpu_eff_median")

    fig2 = go.Figure()
    fig2.add_trace(go.Bar(
        y=[truncate(r) for r in multi_stats["Rule"]],
        x=multi_stats["ncpus"],
        name="Requested CPUs",
        orientation="h",
        marker_color="lightsteelblue",
        customdata=multi_stats["Rule"],
        hovertemplate="<b>%{customdata}</b><br>Requested: %{x} CPUs<extra></extra>",
    ))
    fig2.add_trace(go.Bar(
        y=[truncate(r) for r in multi_stats["Rule"]],
        x=multi_stats["effective_cpus_median"].round(1),
        name="Effective CPUs (median)",
        orientation="h",
        marker_color="steelblue",
        customdata=multi_stats["Rule"],
        hovertemplate="<b>%{customdata}</b><br>Effective: %{x:.1f} CPUs<extra></extra>",
    ))
    fig2.update_layout(
        barmode="overlay",
        height=max(300, len(multi_stats) * 40 + 100),
        template="plotly_white",
        title="Multi-CPU Rules: Requested vs Effective CPUs Used",
        xaxis_title="Number of CPUs",
        legend=dict(orientation="h", yanchor="bottom", y=1.02),
    )
    fig2.show()

    print("Multi-CPU rules summary:")
    for _, row in multi_stats.iterrows():
        utilization = row["effective_cpus_median"] / row["ncpus"] * 100
        status = "OK" if utilization > 50 else "UNDERUTILIZED"
        print(f"  {row['Rule']:55s} {row['ncpus']:2.0f} CPUs requested, "
              f"{row['effective_cpus_median']:.1f} effective ({utilization:.0f}%) [{status}]  "
              f"({row['count']:.0f} jobs)")
else:
    print("No multi-CPU rules found in the data.")

# --- Plot 3: Total wasted CPU-hours per rule ---
top_waste = cpu_stats[cpu_stats["count"] >= 3].nlargest(15, "wasted_cpu_hours").iloc[::-1]
fig3 = go.Figure()
fig3.add_trace(go.Bar(
    y=[truncate(r) for r in top_waste["Rule"]],
    x=top_waste["wasted_cpu_hours"],
    orientation="h",
    marker_color=[
        "firebrick" if v > 10 else "darkorange" if v > 2 else "gold"
        for v in top_waste["wasted_cpu_hours"]
    ],
    customdata=np.column_stack([top_waste["Rule"], top_waste["count"], top_waste["cpu_eff_median"].round(1)]),
    hovertemplate="<b>%{customdata[0]}</b><br>Wasted: %{x:.1f} CPU-hours<br>"
                  "Jobs: %{customdata[1]}<br>Median CPU Eff: %{customdata[2]}%<extra></extra>",
))
fig3.update_layout(
    height=500,
    template="plotly_white",
    title="Top 15 Rules by Total Wasted CPU-Hours (min 3 jobs)",
    xaxis_title="Wasted CPU-Hours",
    showlegend=False,
)
fig3.show()

total_wasted_h = cpu_stats["wasted_cpu_hours"].sum()
print(f"\nTotal wasted CPU-hours across all rules: {total_wasted_h:,.1f} h")
print(f"Top 5 wasteful rules account for {cpu_stats.nlargest(5, 'wasted_cpu_hours')['wasted_cpu_hours'].sum():,.1f} h "
      f"({cpu_stats.nlargest(5, 'wasted_cpu_hours')['wasted_cpu_hours'].sum() / total_wasted_h * 100:.0f}% of total)")

## Runtime Analysis

In [ ]:
runtime_stats = df.groupby("Rule")["Elapsed_sec"].agg(["mean", "median", "std", "min", "max", "count"]).reset_index()
runtime_stats["cv"] = runtime_stats["std"] / runtime_stats["mean"]
runtime_stats = runtime_stats[runtime_stats["count"] >= 5].sort_values("cv", ascending=False)

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "Rules with Highest Runtime Variance (min 5 jobs)",
    "Slowest Rules by Median Runtime (min 5 jobs)"
), horizontal_spacing=0.3)

# Runtime variance
top_var = runtime_stats.head(15).iloc[::-1]
fig.add_trace(go.Bar(
    y=[truncate(r) for r in top_var["Rule"]],
    x=top_var["cv"], orientation="h",
    marker_color="mediumpurple", name="CV",
    customdata=top_var["Rule"],
    hovertemplate="<b>%{customdata}</b><br>CV: %{x:.2f}<extra></extra>"
), row=1, col=1)
fig.add_vline(x=1.0, line_dash="dash", line_color="red", opacity=0.5, row=1, col=1,
    annotation_text="CV=1.0")
fig.update_xaxes(title_text="Coefficient of Variation", row=1, col=1)

# Slowest rules
top_slow = runtime_stats.nlargest(15, "median").iloc[::-1]
fig.add_trace(go.Bar(
    y=[truncate(r) for r in top_slow["Rule"]],
    x=top_slow["median"] / 60, orientation="h",
    marker_color="coral", name="Runtime",
    customdata=top_slow["Rule"],
    hovertemplate="<b>%{customdata}</b><br>Median: %{x:.1f} min<extra></extra>"
), row=1, col=2)
fig.update_xaxes(title_text="Median Runtime (minutes)", row=1, col=2)

fig.update_layout(height=600, showlegend=False, template="plotly_white")
fig.show()

print("\nRules with high runtime variance (CV > 1.0) — candidates for dynamic resources:")
high_cv = runtime_stats[runtime_stats["cv"] > 1.0]
if len(high_cv) > 0:
    for _, row in high_cv.iterrows():
        print(f"  {row['Rule']:50s} CV={row['cv']:.2f}  median={row['median']/60:.1f}min  range=[{row['min']/60:.1f}, {row['max']/60:.1f}]min")
else:
    print("  None found.")

## Runtime Allocation Analysis

Compares actual elapsed time against the SLURM time limit (default: 600 min / 10 hours for all rules).
Shows how over-provisioned time limits are and suggests tighter limits to improve scheduler backfill.

In [ ]:
SLURM_TIME_LIMIT_MIN = 600  # Default time limit from SLURM profile (10 hours)
RUNTIME_SAFETY_MARGIN = 2.0  # 2x headroom above p95 for suggested time limits
MIN_RUNTIME_MIN = 10  # Minimum suggested time limit

# Per-rule runtime stats for allocation analysis
rt_alloc = df.groupby("Rule").agg(
    count=("Rule", "size"),
    elapsed_median_sec=("Elapsed_sec", "median"),
    elapsed_p95_sec=("Elapsed_sec", lambda x: x.quantile(0.95)),
    elapsed_max_sec=("Elapsed_sec", "max"),
).reset_index()

rt_alloc["elapsed_median_min"] = rt_alloc["elapsed_median_sec"] / 60
rt_alloc["elapsed_p95_min"] = rt_alloc["elapsed_p95_sec"] / 60
rt_alloc["elapsed_max_min"] = rt_alloc["elapsed_max_sec"] / 60
rt_alloc["time_limit_min"] = SLURM_TIME_LIMIT_MIN
rt_alloc["utilization_pct"] = (rt_alloc["elapsed_p95_min"] / SLURM_TIME_LIMIT_MIN) * 100
rt_alloc["suggested_limit_min"] = np.maximum(
    rt_alloc["elapsed_p95_min"] * RUNTIME_SAFETY_MARGIN, MIN_RUNTIME_MIN
).round(0)
rt_alloc["unused_time_per_job_min"] = SLURM_TIME_LIMIT_MIN - rt_alloc["elapsed_median_min"]
rt_alloc["total_unused_time_hours"] = (rt_alloc["unused_time_per_job_min"] * rt_alloc["count"]) / 60
rt_alloc = rt_alloc.sort_values("total_unused_time_hours", ascending=False)

# --- Plot 1: Runtime utilization as % of time limit ---
freq_rt = rt_alloc[rt_alloc["count"] >= 5].copy()
freq_rt_sorted = freq_rt.sort_values("utilization_pct", ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(
    y=[truncate(r) for r in freq_rt_sorted["Rule"]],
    x=freq_rt_sorted["utilization_pct"],
    orientation="h",
    marker_color=[
        "seagreen" if v > 10 else "gold" if v > 1 else "firebrick"
        for v in freq_rt_sorted["utilization_pct"]
    ],
    customdata=np.column_stack([
        freq_rt_sorted["Rule"],
        freq_rt_sorted["elapsed_p95_min"].round(1),
        freq_rt_sorted["elapsed_max_min"].round(1),
    ]),
    hovertemplate="<b>%{customdata[0]}</b><br>"
                  "Time used (p95): %{customdata[1]} min of 600 min<br>"
                  "Max observed: %{customdata[2]} min<extra></extra>",
))
fig.add_vline(x=10, line_dash="dash", line_color="red", opacity=0.5,
    annotation_text="10% utilization")
fig.update_layout(
    height=max(500, len(freq_rt) * 25 + 100),
    template="plotly_white",
    title=f"Runtime Utilization: p95 Elapsed as % of {SLURM_TIME_LIMIT_MIN}-min Time Limit (min 5 jobs)",
    xaxis_title="Time Limit Utilization (%)",
    yaxis=dict(dtick=1),
    showlegend=False,
)
fig.show()

# --- Plot 2: Suggested time limits vs current ---
top_overprovisioned = freq_rt.nlargest(20, "total_unused_time_hours").sort_values("suggested_limit_min")

fig2 = go.Figure()
fig2.add_trace(go.Bar(
    y=[truncate(r) for r in top_overprovisioned["Rule"]],
    x=[SLURM_TIME_LIMIT_MIN] * len(top_overprovisioned),
    name=f"Current Limit ({SLURM_TIME_LIMIT_MIN} min)",
    orientation="h",
    marker_color="lightcoral",
    opacity=0.4,
))
fig2.add_trace(go.Bar(
    y=[truncate(r) for r in top_overprovisioned["Rule"]],
    x=top_overprovisioned["suggested_limit_min"],
    name="Suggested Limit (p95 × 2)",
    orientation="h",
    marker_color="seagreen",
    customdata=np.column_stack([
        top_overprovisioned["Rule"],
        top_overprovisioned["elapsed_p95_min"].round(1),
        top_overprovisioned["suggested_limit_min"].astype(int),
    ]),
    hovertemplate="<b>%{customdata[0]}</b><br>"
                  "p95 runtime: %{customdata[1]} min<br>"
                  "Suggested limit: %{customdata[2]} min<extra></extra>",
))
fig2.add_trace(go.Scatter(
    y=[truncate(r) for r in top_overprovisioned["Rule"]],
    x=top_overprovisioned["elapsed_max_min"],
    mode="markers",
    name="Max Observed",
    marker=dict(color="black", size=8, symbol="diamond"),
    customdata=top_overprovisioned["Rule"],
    hovertemplate="<b>%{customdata}</b><br>Max: %{x:.1f} min<extra></extra>",
))
fig2.update_layout(
    barmode="overlay",
    height=max(400, len(top_overprovisioned) * 30 + 100),
    template="plotly_white",
    title="Top 20 Over-Provisioned Rules: Current vs Suggested Time Limits",
    xaxis_title="Time (minutes)",
    xaxis_type="log",
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)
fig2.show()

# Summary
total_unused_h = rt_alloc["total_unused_time_hours"].sum()
max_rule = rt_alloc.iloc[0]
print(f"\nTotal unused scheduler time across all jobs: {total_unused_h:,.0f} hours")
print(f"  (= {total_unused_h / 24:,.0f} node-days of idle time reservations)")
print(f"\nAll {len(rt_alloc)} rules use <{rt_alloc['elapsed_max_min'].max():.0f} min — "
      f"the 600-min default is {SLURM_TIME_LIMIT_MIN / rt_alloc['elapsed_max_min'].max():.0f}x the longest observed job.")
print(f"\nTop 5 rules by wasted scheduler time:")
for _, row in rt_alloc.head(5).iterrows():
    print(f"  {row['Rule']:50s} {row['count']:4.0f} jobs × ~{row['unused_time_per_job_min']:.0f} min unused "
          f"= {row['total_unused_time_hours']:.0f} h  (suggest: {row['suggested_limit_min']:.0f} min)")

print(f"\nNote: While SLURM does not charge for unused time on most clusters, tighter time limits")
print(f"improve scheduler backfill efficiency — shorter jobs can fill gaps in the queue.")

## Recommendations Summary

In [ ]:
recs = mem_opt[mem_opt["count"] >= 3].copy()
recs = recs.merge(rule_stats[["Rule", "cpu_eff_median", "elapsed_median", "elapsed_p95"]], on="Rule")

# Merge CPU stats (ncpus_mode, wasted_cpu_hours)
recs = recs.merge(
    cpu_stats[["Rule", "ncpus_mode", "wasted_cpu_hours"]],
    on="Rule", how="left"
)

# Merge runtime allocation suggestions
recs = recs.merge(
    rt_alloc[["Rule", "elapsed_p95_min", "elapsed_max_min", "suggested_limit_min"]],
    on="Rule", how="left"
)

def full_recommendation(row):
    parts = []
    # Memory
    if row["savings_per_job_MB"] > 100:
        parts.append(f"Mem: {row['mem_requested_mean']:.0f} -> {row['suggested_MB']:.0f} MB")
    # CPU efficiency
    if row["cpu_eff_median"] < 15 and row.get("ncpus_mode", 1) > 1:
        parts.append(f"CPU: {row['cpu_eff_median']:.0f}% eff on {row['ncpus_mode']:.0f} cores — reduce threads")
    elif row["cpu_eff_median"] < 15:
        parts.append(f"CPU: {row['cpu_eff_median']:.0f}% eff — I/O bound or short job")
    # Runtime
    suggested = row.get("suggested_limit_min", np.nan)
    if pd.notna(suggested) and suggested < SLURM_TIME_LIMIT_MIN * 0.5:
        parts.append(f"Time: {SLURM_TIME_LIMIT_MIN} -> {suggested:.0f} min")
    if not parts:
        parts.append("OK")
    return "; ".join(parts)

recs["recommendation"] = recs.apply(full_recommendation, axis=1)
recs = recs[recs["recommendation"] != "OK"].sort_values("total_savings_MB", ascending=False)

print(f"Rules with optimization opportunities: {len(recs)}\n")

# Build cell colors
savings_max = recs["savings_per_job_MB"].max() if len(recs) > 0 else 1
savings_colors = []
for v in recs["savings_per_job_MB"]:
    if v > 0:
        intensity = min(200, max(50, int(v / savings_max * 200)))
        savings_colors.append(f"rgba(0,{intensity},0,0.15)")
    else:
        savings_colors.append("white")

cpu_vals = recs["cpu_eff_median"].values
cpu_colors = ["rgba(255,50,50,0.2)" if v < 15 else "white" for v in cpu_vals]

suggested_limits = recs["suggested_limit_min"].fillna(SLURM_TIME_LIMIT_MIN).values
time_colors = [
    "rgba(255,165,0,0.2)" if v < SLURM_TIME_LIMIT_MIN * 0.1 else "white"
    for v in suggested_limits
]

fig = go.Figure(data=[go.Table(
    columnwidth=[220, 45, 70, 70, 70, 55, 55, 65, 300],
    header=dict(
        values=["Rule", "Jobs", "Current<br>Mem (MB)", "Suggested<br>Mem (MB)",
                "Savings/Job<br>(MB)", "CPU<br>Eff %", "CPUs", "Suggested<br>Time (min)",
                "Recommendation"],
        fill_color="#2c3e50", font=dict(color="white", size=11),
        align="left"
    ),
    cells=dict(
        values=[
            recs["Rule"].values,
            recs["count"].values,
            recs["mem_requested_mean"].round(0).astype(int).values,
            recs["suggested_MB"].round(0).astype(int).values,
            recs["savings_per_job_MB"].round(0).astype(int).values,
            (recs["cpu_eff_median"].round(1).astype(str) + "%").values,
            recs["ncpus_mode"].fillna(1).astype(int).values,
            recs["suggested_limit_min"].fillna(SLURM_TIME_LIMIT_MIN).astype(int).values,
            recs["recommendation"].values,
        ],
        fill_color=[
            "white", "white", "white", "white",
            savings_colors,
            cpu_colors,
            "white",
            time_colors,
            "white"
        ],
        align="left", font=dict(size=11)
    )
)])
fig.update_layout(
    title="Actionable Recommendations (Memory, CPU & Runtime)",
    height=max(400, 30 * len(recs) + 100),
    template="plotly_white"
)
fig.show()